In [4]:
!git clone https://github.com/meta-compute/sem-fracture-deep-learning.git

Cloning into 'sem-fracture-deep-learning'...
remote: Enumerating objects: 2691, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 2691 (delta 0), reused 3 (delta 0), pack-reused 2685 (from 2)
Receiving objects: 100% (2691/2691), 904.02 MiB | 40.42 MiB/s, done.
Resolving deltas: 100% (17/17), done.
Updating files: 100% (2000/2000), done.
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [10]:
cd sem-fracture-deep-learning

/kaggle/working/sem-fracture-deep-learning


In [12]:
!git checkout vishnu-dev
!pip install -r requirements.txt

Updating files: 100% (635/635), done.
branch 'vishnu-dev' set up to track 'origin/vishnu-dev'.
Switched to a new branch 'vishnu-dev'


# Fracture modelling

Use the sections below in order: Train, Validate, then optionally Test. Mixed augmentation is training-only. Final test evaluation is disabled until explicitly enabled. The stored training output is from an earlier interrupted attempt.


## Train

Configure the model in the setup cell and run training. The best validation-macro-F1 checkpoint is saved under `results/shared_pipeline/`.


In [26]:
from pathlib import Path

import pandas as pd
import torch

from sem_fracture.pipeline import run_experiment, run_prediction

DATA_DIR = Path.cwd()
MODEL_KEY = "tinyvit_21m_dual_attention"  # resnet18, deit_tiny, tinyvit_11m, tinyvit_21m, tinyvit_21m_dual_attention
MODE = "head_only"  # head_only or finetune
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

BASELINE_CHECKPOINTS = {
    "resnet18": DATA_DIR / "results" / "resnet18_best.pth",
    "deit_tiny": DATA_DIR / "results" / "deit_tiny" / "checkpoints" / "deit_tiny_best.pth",
    "tinyvit_11m": DATA_DIR / "results" / "tinyvit_11m" / "checkpoints" / "tinyvit_11m_best.pth",
    "tinyvit_21m": DATA_DIR / "results" / "tinyvit_21m" / "checkpoints" / "tinyvit_21m_best.pth",
    "tinyvit_21m_dual_attention": DATA_DIR / "results" / "tinyvit_21m_finetune" / "checkpoints" / "tinyvit_21m_finetuned_best.pth",
}

if MODE not in ("head_only", "finetune"):
    raise ValueError(f"Unknown training mode: {MODE}")
if MODEL_KEY not in BASELINE_CHECKPOINTS:
    raise ValueError(f"Unknown model key: {MODEL_KEY}")

IS_DUAL_ATTENTION = MODEL_KEY == "tinyvit_21m_dual_attention"
INITIAL_CHECKPOINT = (
    BASELINE_CHECKPOINTS[MODEL_KEY] if (MODE == "finetune" or IS_DUAL_ATTENTION) else None
)
BACKBONE_LR = 1e-5 if (MODE == "finetune" and not IS_DUAL_ATTENTION) else None
HEAD_LR = 1e-4 if (MODE == "finetune" or IS_DUAL_ATTENTION) else 1e-3
EPOCHS = 30 if (MODEL_KEY.startswith("tinyvit_21m") and (MODE == "finetune" or IS_DUAL_ATTENTION)) else 10
PATIENCE = 4 if (MODE == "finetune" or IS_DUAL_ATTENTION) else None
GRAD_CLIP_NORM = 1.0 if (MODEL_KEY.startswith("tinyvit") and MODE == "finetune" and not IS_DUAL_ATTENTION) else None
AUGMENT_MIXED = False if IS_DUAL_ATTENTION else True
CHECKPOINT_PATH = (
    DATA_DIR / "results" / "shared_pipeline"
    / MODEL_KEY / "class_balanced" / f"{MODEL_KEY}_{MODE}_best.pth"
)
RUN_FINAL_TEST_EVALUATION = False  # enable only after selecting the checkpoint


In [27]:
config = {
    "DATA_DIR": str(DATA_DIR),
    "MODEL_KEY": MODEL_KEY,
    "MODE": MODE,
    "DEVICE": str(DEVICE),
    "IS_DUAL_ATTENTION": IS_DUAL_ATTENTION,
    "INITIAL_CHECKPOINT": str(INITIAL_CHECKPOINT) if INITIAL_CHECKPOINT else None,
    "BACKBONE_LR": BACKBONE_LR,
    "HEAD_LR": HEAD_LR,
    "EPOCHS": EPOCHS,
    "PATIENCE": PATIENCE,
    "GRAD_CLIP_NORM": GRAD_CLIP_NORM,
    "AUGMENT_MIXED": AUGMENT_MIXED,
    "CHECKPOINT_PATH": str(CHECKPOINT_PATH),
    "RUN_FINAL_TEST_EVALUATION": RUN_FINAL_TEST_EVALUATION,
}

for key, value in config.items():
    print(f"{key}: {value}")

DATA_DIR: /kaggle/working/sem-fracture-deep-learning
MODEL_KEY: tinyvit_21m_dual_attention
MODE: head_only
DEVICE: cuda
IS_DUAL_ATTENTION: True
INITIAL_CHECKPOINT: /kaggle/working/sem-fracture-deep-learning/results/tinyvit_21m_finetune/checkpoints/tinyvit_21m_finetuned_best.pth
BACKBONE_LR: None
HEAD_LR: 0.0001
EPOCHS: 30
PATIENCE: 4
GRAD_CLIP_NORM: None
AUGMENT_MIXED: False
CHECKPOINT_PATH: /kaggle/working/sem-fracture-deep-learning/results/shared_pipeline/tinyvit_21m_dual_attention/class_balanced/tinyvit_21m_dual_attention_head_only_best.pth
RUN_FINAL_TEST_EVALUATION: False


In [28]:
result = run_experiment(
    data_dir=DATA_DIR,
    model_key=MODEL_KEY,
    mode=MODE,
    checkpoint_path=CHECKPOINT_PATH,
    device=DEVICE,
    head_lr=HEAD_LR,
    backbone_lr=BACKBONE_LR,
    weight_decay=1e-4,
    epochs=EPOCHS,
    patience=PATIENCE,
    grad_clip_norm=GRAD_CLIP_NORM,
    initial_checkpoint=INITIAL_CHECKPOINT,
    augment_mixed=False,
    loss_type="class_balanced",
    seed=42,
    batch_size=64,
)

best = result["best_checkpoint"]
print("Best epoch:", best["epoch"])
print("Best validation macro-F1:", f"{best['val_macro_f1']:.4f}")
print("Checkpoint:", CHECKPOINT_PATH)
print("Training split:", result["split_file"])
print("Class weights:", result["class_weights"])

Epoch 1/30 starting
Epoch 1/30 | train_loss=0.6462 | val_loss=0.5842 | val_accuracy=0.6071 | val_macro_f1=0.5194 | elapsed=0.2s | best_checkpoint=True
Epoch 2/30 starting
Epoch 2/30 | train_loss=0.5336 | val_loss=0.4844 | val_accuracy=0.7714 | val_macro_f1=0.6776 | elapsed=0.3s | best_checkpoint=True
Epoch 3/30 starting
Epoch 3/30 | train_loss=0.4447 | val_loss=0.4081 | val_accuracy=0.8214 | val_macro_f1=0.7335 | elapsed=0.3s | best_checkpoint=True
Epoch 4/30 starting
Epoch 4/30 | train_loss=0.3767 | val_loss=0.3484 | val_accuracy=0.8714 | val_macro_f1=0.7862 | elapsed=0.2s | best_checkpoint=True
Epoch 5/30 starting
Epoch 5/30 | train_loss=0.3262 | val_loss=0.3014 | val_accuracy=0.8964 | val_macro_f1=0.8140 | elapsed=0.2s | best_checkpoint=True
Epoch 6/30 starting
Epoch 6/30 | train_loss=0.2893 | val_loss=0.2641 | val_accuracy=0.9179 | val_macro_f1=0.8409 | elapsed=0.2s | best_checkpoint=True
Epoch 7/30 starting
Epoch 7/30 | train_loss=0.2571 | val_loss=0.2325 | val_accuracy=0.9393 | v

## Validate

Evaluate the selected checkpoint on the original validation split. Validation data is not augmented.


In [32]:
validation_result = run_prediction(
    data_dir=DATA_DIR,
    model_key=MODEL_KEY,
    checkpoint_path=CHECKPOINT_PATH,
    split="validation",
    split_file="dataset_split.csv",
    device=DEVICE,
    batch_size=16,
)

print(validation_result["metrics"])
pd.DataFrame(validation_result["predictions"])

{'confusion_matrix': [[115, 0, 2], [0, 137, 5], [0, 2, 19]], 'accuracy': 0.9678571428571429, 'precision': [1.0, 0.9856115107913669, 0.7307692307692307], 'recall': [0.9829059829059829, 0.9647887323943662, 0.9047619047619048], 'class_f1': [0.9913793103448275, 0.9750889679715302, 0.8085106382978723], 'macro_f1': 0.9249929722047433}


,filename,actual,predicted,correct
0,100Cr6_AR_06_10.png,Brittle,Mixed,False
1,100Cr6_AR_06_3.png,Brittle,Brittle,True
2,304_H2_02_1.png,Brittle,Brittle,True
3,304_H2_02_10.png,Brittle,Brittle,True
4,304_H2_02_11.png,Brittle,Brittle,True
...,...,...,...,...
275,mX65_H2_29_22.png,Mixed,Mixed,True
276,mX65_H2_29_23.png,Mixed,Mixed,True
277,mX65_H2_29_3.png,Mixed,Mixed,True
278,mX65_H2_29_31.png,Mixed,Mixed,True


## Test

After selecting the checkpoint with validation, set `RUN_FINAL_TEST_EVALUATION = True` in the setup cell and run the cell below once. It uses the original `dataset_split.csv`; generated Mixed images belong to training only.


In [30]:
RUN_FINAL_TEST_EVALUATION = True

In [33]:
if RUN_FINAL_TEST_EVALUATION:
    final_test_result = run_prediction(
        data_dir=DATA_DIR,
        model_key=MODEL_KEY,
        checkpoint_path=CHECKPOINT_PATH,
        split="test",
        split_file="dataset_split.csv",
        device=DEVICE,
        batch_size=16,
    )
    print("Final held-out test metrics:", final_test_result["metrics"])
else:
    print("Final test evaluation is disabled.")


Final held-out test metrics: {'confusion_matrix': [[114, 0, 2], [0, 130, 13], [2, 1, 17]], 'accuracy': 0.9354838709677419, 'precision': [0.9827586206896551, 0.9923664122137404, 0.53125], 'recall': [0.9827586206896551, 0.9090909090909091, 0.85], 'class_f1': [0.9827586206896551, 0.948905109489051, 0.6538461538461537], 'macro_f1': 0.8618366280082866}
